In [9]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import statsmodels.api as sm

from src.data import *
from src.models import *
from src.features import log_returns, realised_volatility
from src.backtest import walk_forward
from src.metrics import compare_models, dm_table

### ==== Validation =====

In [2]:
spot = load_brent_spot(start="1987-01-01")
data = spot.copy()
data["log_return"] = log_returns(data, col="brent_spot")
data = data.dropna()


val_args = dict(horizon=60, start_date="2024-01-01", end_date="2025-12-31", target_end_date="2025-12-31")


val_results = {
    "Random Walk": walk_forward(data, forecast_funct=random_walk_forecast, **val_args),
    "Historical Mean": walk_forward(data, forecast_funct=historical_mean_forecast, lookback=3000, **val_args),
    "AR": walk_forward(data, forecast_funct=ar1_forecast, lookback=3000, **val_args),
    "Direct forecast": walk_forward(data, forecast_funct=direct_forecast, lookback=3000, **val_args),
    "Direct lag_1 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=1, long=60, lookback=3000, **val_args),
    "Direct lag_5 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=5, long=60, lookback=3000, **val_args)
}

compare_models(val_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,5.289485,6.623020,7.328490,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,5.154657,6.483503,7.110097,0.691275,0.000000,0.000000,0.000000,0.691275,1.000000,0.817460,0.089103,0.364276,-0.010277,1.0
AR,5.154056,6.481886,7.109355,0.686801,0.000000,0.000000,0.000000,0.689888,0.993528,0.814324,0.089103,0.364276,-0.010277,1.0
Direct forecast,4.500004,5.786758,6.251301,0.760626,0.574879,0.862319,0.689855,0.920833,0.715210,0.805100,0.219730,1.533983,-0.012407,11.0
Direct lag_1 lag_60,4.491483,5.773978,6.239654,0.756152,0.570732,0.847826,0.682216,0.913223,0.715210,0.802178,0.219730,1.533983,-0.012407,11.0
Direct lag_5 lag_60,4.582147,5.881958,6.361988,0.747204,0.561576,0.826087,0.668622,0.901639,0.711974,0.795660,0.072065,0.255744,-0.159789,11.0


#### Mean Revert

In [ ]:
# PRICES = ["brent_spot", "wti", "usd", "sp500"]   # prices -> log returns
# INDICES = ["vix", "ovx", "epu", "gpr"]           # fear/uncertainty indices -> log levels

df_mr = load_dataset()
df_mr = df_mr[["brent_spot", "wti"]]
df_mr = df_mr.rename(columns={"wti": "wti_spot"})
df_mr["log_return"] = np.log(df_mr["brent_spot"]).diff()
df_mr = df_mr.dropna()

df_mr.tail()

In [ ]:
val_forecast = walk_forward(df_mr, forecast_funct=spread_mr_forecast, lookback=3000, **val_args)
val_forecast.tail()

In [ ]:
val_results["Mean Revert"] = val_forecast

compare_models(val_results, horizon=60)

#### XGBoost

In [ ]:
val_forecast = walk_forward(data, forecast_funct=xgb_direct_forecast, lookback=3000, **val_args)
val_forecast.tail()

In [ ]:
val_results["XGBoost"] = val_forecast

compare_models(val_results, horizon=60)

#### Direct Ridge Forecast

In [ ]:
df_rr = load_dataset()
  
df_rr = df_rr[["brent_spot", "wti", "usd", "sp500", "vix", "ovx"]]
df_rr = df_rr.rename(columns={"wti": "wti_spot"})
df_rr = df_rr.dropna()

for col in ["usd", "sp500", "vix", "ovx"]:
    df_rr[col] = np.log(df_rr[col])

df_rr.tail()

In [ ]:
val_forecast = walk_forward(df_rr, forecast_funct=direct_ridge_forecast, lookback=3000, **val_args)
val_forecast.tail()

In [ ]:
val_results["Direct ridge"] = val_forecast

compare_models(val_results,  horizon=60)

#### Combination Forecast

In [10]:
weights = [0.1, 0.35, 0.2, 0.35]
val_forecast = walk_forward(data, forecast_funct=combination_forecast, weights=weights, **val_args)
val_forecast.tail()

NameError: name 'combination_forecast' is not defined

In [ ]:
val_results["Combination"] = val_forecast

compare_models(val_results, horizon=60)

### ==== Diebold-Mariano test ====

In [ ]:
for loss in ["ae", "se"]:
    print(f"\nDM test vs Random Walk, loss = {loss}")
    print(dm_table(val_results, benchmark="Random Walk", horizon=60, loss=loss).round(4))

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(np.array(val_results["Random Walk"]["target_date"]), np.array(val_results["Random Walk"]["actual"]), label='Actual', color='black', lw=1)

for name in ["Random Walk", "Combination"]:
    r = val_results[name]
    plt.plot(np.array(r["target_date"]), np.array(r["forecast"]), label=name, alpha=0.8, lw=0.8)

plt.title("60-Day test: Target-dated prices ($/barrel)")
plt.legend(loc="upper right")
plt.show()

### ==== Test ====

In [ ]:
test_args = dict(horizon=60, start_date="2026-01-01", end_date="2026-09-30", target_end_date="2026-09-30")

test_results = {
    "Random Walk": walk_forward(data, forecast_funct=random_walk_forecast, **test_args),
    "Historical Mean": walk_forward(data, forecast_funct=historical_mean_forecast, lookback=3000, **test_args),
    "AR": walk_forward(data, forecast_funct=ar1_forecast, lookback=3000, **test_args),
    "Direct forecast": walk_forward(data, forecast_funct=direct_forecast, lookback=3000, **test_args),
    "Direct lag_1 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=1, long=60, lookback=3000, **test_args),
    "Direct lag_5 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=5, long=60, lookback=3000, **test_args),
    "Mean revert": walk_forward(df_mr, forecast_funct=spread_mr_forecast, lookback=3000, **test_args),
    "XGBoost": walk_forward(data, forecast_funct=xgb_direct_forecast, lookback=3000, **test_args),
    "Direct ridge": walk_forward(df_rr, forecast_funct=direct_ridge_forecast, lookback=3000, **test_args),
    "Combination": walk_forward(data, forecast_funct=combination_forecast, weights=weights, **test_args)
}

compare_models(test_results, horizon=60)

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(np.array(test_results["Random Walk"]["target_date"]), np.array(test_results["Random Walk"]["actual"]), label='Actual', color='black', lw=1)

for name in ["Random Walk", "Combination"]:
    r = test_results[name]
    plt.plot(np.array(r["target_date"]), np.array(r["forecast"]), label=name, alpha=0.8, lw=0.8)

plt.title("60-Day test: Target-dated prices ($/barrel)")
plt.legend(loc="upper right")
plt.show()

### ==== Final 60-day forecast =====
origin = 2026-09-30

In [ ]:
ORIGIN_DATE = "2026-09-30"
HORIZON, LOOKBACK = 60, 3000        # 60 trading days ahead, 3000-day fitting window
SHORT, LONG = 1, 60                 # r_1d and lagged r_60d (non-overlapping, days t-59..t-1)

# 1) Locate the origin (last available trading day on or before ORIGIN_DATE)
dates = data.index
origin_idx = dates.searchsorted(pd.Timestamp(ORIGIN_DATE), side="right") - 1
origin_date = dates[origin_idx]
if origin_date != pd.Timestamp(ORIGIN_DATE):
    print(f"WARNING: {ORIGIN_DATE} not in data, using {origin_date.date()} instead")
fit_idx = origin_idx                # model fitted only on targets known by the origin
start = fit_idx - LOOKBACK + 1
assert start >= 0, "not enough history"

# 2) Rebuild the training set exactly as direct_lags_forecast does (so we can inspect the fit)
lp = np.log(data["brent_spot"].to_numpy(float))        # log of PRICE (not of log_return)

def make_X(t):
    return pd.DataFrame({
        "r_1d":         lp[t] - lp[t - SHORT],         # latest 1-day log return
        "r_60d_lagged": lp[t - SHORT] - lp[t - LONG],  # log return over t-59..t-1
    })

t = np.arange(start + LONG, fit_idx - HORIZON + 1)     # last target = fit_idx
X = make_X(t)
y = pd.Series(lp[t + HORIZON] - lp[t], name="ret_60d_ahead")

fit = sm.OLS(y.values, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": HORIZON})

# 3) Parameters and significance (Newey-West/HAC, because 60-day targets overlap)
print(fit.summary())
ci = np.asarray(fit.conf_int())
names = fit.model.exog_names
params = pd.DataFrame({"coef": np.asarray(fit.params), "HAC se": np.asarray(fit.bse),
                       "t": np.asarray(fit.tvalues), "p-value": np.asarray(fit.pvalues),
                       "95% low": ci[:, 0], "95% high": ci[:, 1]}, index=names)
print("\n", params.round(4))
print(f"\nobs = {len(y)}  (~{len(y)//HORIZON} non-overlapping 60-day windows)")
print(f"R2 = {fit.rsquared:.4f}   joint Wald p-value = {fit.f_pvalue:.4f}")

# 4) Forecast from the origin
x0 = np.r_[1.0, make_X(np.array([origin_idx])).to_numpy()[0]]
beta = np.asarray(fit.params)
pred_ret = float(x0 @ beta)
se_mean = float(np.sqrt(x0 @ np.asarray(fit.cov_params()) @ x0))   # uncertainty of expected return
sigma = float(np.sqrt(fit.scale))                       # residual sd of a 60-day return
sd_total = np.sqrt(se_mean**2 + sigma**2)
p0 = data["brent_spot"].iloc[origin_idx]
f = lambda r: p0 * np.exp(r)

# sanity check against the function used in the backtest
fc_fn = direct_lags_forecast(data, origin_idx, horizon=HORIZON, fit_idx=fit_idx,
                             lookback=LOOKBACK, short=SHORT, long=LONG,
                             nonoverlap=True, use_intercept=True)
assert np.isclose(f(pred_ret), fc_fn), "cell and model function disagree"

target_date = pd.bdate_range(origin_date, periods=HORIZON + 1)[-1]   # approx: ignores holidays
print(f"\nOrigin {origin_date.date()}  price = {p0:.2f}")
print(f"Features: r_1d = {x0[1]:+.4f}, r_60d_lagged = {x0[2]:+.4f}")
print(f"Predicted 60-day log return = {pred_ret:+.4f}  (expected-return 95% CI "
      f"{pred_ret-1.96*se_mean:+.4f} to {pred_ret+1.96*se_mean:+.4f})")
print(f"Point forecast for ~{target_date.date()}: {f(pred_ret):.2f}")
print(f"95% prediction interval: {f(pred_ret-1.96*sd_total):.2f} to {f(pred_ret+1.96*sd_total):.2f}")
print(f"Random-walk forecast: {p0:.2f}")

# 5) Optional: how the forecast is doing so far, if data already runs past the origin
elapsed = len(data) - 1 - origin_idx
if elapsed > 0:
    print(f"\n{elapsed} trading days elapsed. Latest {data.index[-1].date()}: "
          f"{data['brent_spot'].iloc[-1]:.2f} (realised log return {lp[-1]-lp[origin_idx]:+.4f})")

#### GARCH-based prediction interval

In [ ]:
from arch import arch_model

def garch_interval_forecast(df, origin_idx, horizon=60, fit_idx=None, lookback=3000,
                            center_fn=None, level=0.95, vol="garch", gjr=False):
    """Prediction interval for Brent at origin+horizon.
    - centre : log-return forecast of `center_fn` (default: random walk, 0 return)
    - width  : sqrt of the summed h-step GARCH(1,1)-t variance forecasts
               (vol="constant" gives the benchmark sd*sqrt(h) from the fit window)
    - params are estimated on data up to fit_idx only; the volatility state is then
      filtered up to origin_idx with params frozen (no look-ahead).
    Returns dict(forecast, lower, upper, sigma_h)."""
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"
    start = fit_idx - lookback + 1
    assert start >= 0, "not enough history for the lookback window"

    r = df["log_return"].to_numpy(float)
    assert not np.isnan(r[start: origin_idx + 1]).any(), "NaNs in return window"
    p_t = df["brent_spot"].iloc[origin_idx]

    used = vol
    if vol == "garch":
        spec = dict(mean="Zero", vol="GARCH", p=1, o=int(gjr), q=1, dist="t", rescale=False)
        fit = arch_model(r[start: fit_idx + 1] * 100, **spec).fit(disp="off")   # pct scale
        if fit.convergence_flag != 0:
            used = "constant (garch failed)"
        else:
            filt = arch_model(r[start: origin_idx + 1] * 100, **spec).fix(fit.params)
            var_path = filt.forecast(horizon=horizon, reindex=False).variance.to_numpy()[-1]
            sigma_h = np.sqrt(var_path.sum()) / 100          # sd of the h-day cumulative log return
    if used != "garch":
        sigma_h = r[start: fit_idx + 1].std(ddof=1) * np.sqrt(horizon)

    pred_ret = 0.0 if center_fn is None else np.log(center_fn(df, origin_idx, horizon, fit_idx) / p_t)
    z = stats.norm.ppf(0.5 + level / 2)    # 60-day sum of returns is close to normal even if daily is fat-tailed
    return {"forecast": p_t * np.exp(pred_ret),
            "lower": p_t * np.exp(pred_ret - z * sigma_h),
            "upper": p_t * np.exp(pred_ret + z * sigma_h),
            "sigma_h": sigma_h, "vol_used": used}


def walk_forward_interval(df, horizon, start_date, end_date, interval_fn,
                          target_end_date=None, **kwargs):
    """Same loop as walk_forward but records lower/upper too (result also works with the metric functions)."""
    dates = df.index
    start_idx, end_idx = dates.searchsorted(start_date), dates.searchsorted(end_date)
    rows = []
    for i in range(start_idx, end_idx):
        j = i + horizon
        if j >= len(dates): break
        if target_end_date is not None and dates[j] > pd.Timestamp(target_end_date): break
        out = interval_fn(df, i, horizon, i, **kwargs)
        rows.append({"origin_date": dates[i], "target_date": dates[j],
                     "origin_price": df["brent_spot"].iloc[i], **out,
                     "actual": df["brent_spot"].iloc[j]})
    return pd.DataFrame(rows)


def interval_metrics(res, level=0.95):
    """Coverage should be close to `level`. Interval (Winkler) score: lower = better (rewards narrow AND correct)."""
    a = 1 - level
    inside = (res["actual"] >= res["lower"]) & (res["actual"] <= res["upper"])
    width = res["upper"] - res["lower"]
    score = width + (2 / a) * (res["lower"] - res["actual"]).clip(lower=0) \
                  + (2 / a) * (res["actual"] - res["upper"]).clip(lower=0)
    return {"coverage": inside.mean(), "target": level, "mean_width": width.mean(),
            "mean_rel_width": (width / res["origin_price"]).mean(), "interval_score": score.mean(),
            "n": len(res), "eff_n (n/h)": len(res) / 60}   # overlapping windows: effective n is ~n/h

In [ ]:
from functools import partial

origin_idx = data.index.searchsorted(pd.Timestamp("2026-09-30"), side="right") - 1
point_model = partial(direct_lags_forecast, short=1, long=60, lookback=3000)

garch_interval_forecast(data, origin_idx, horizon=60, fit_idx=origin_idx,
                              lookback=3000, center_fn=point_model, level=0.95)